# 🧿 DeepGuard AI — EfficientNet-B4 Fine-tuning
### FaceForensics++ Deepfake Detection
**Runtime:** GPU (T4 or A100) | **Est. time:** 4-8 hours

In [ ]:
# Step 1: Install dependencies
!pip install timm torch torchvision opencv-python-headless scikit-learn matplotlib seaborn -q
!pip install huggingface-hub -q
print('✅ Dependencies installed')

In [ ]:
# Step 2: Download FaceForensics++ dataset
# IMPORTANT: You must have the download script from https://github.com/ondyari/FaceForensics
# Fill in your credentials below after getting access

import os

# Option A: If you have the download script
# !python faceforensics_download.py . -d all -c c23 -t face_images

# Option B: Use a small public sample for testing
!mkdir -p data/real data/fake

# Download sample faces (CelebA for real, for demo)
!pip install datasets -q
from datasets import load_dataset
ds = load_dataset('nielsr/celeba-faces', split='train[:500]', trust_remote_code=True)

# Save real faces
for i, item in enumerate(ds):
    item['image'].save(f'data/real/real_{i:04d}.jpg')
print(f'✅ Saved {len(ds)} real face images')

In [ ]:
# Step 3: Dataset & DataLoader
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os, glob, random
from sklearn.model_selection import train_test_split

class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Load paths & labels
real = glob.glob('data/real/*.jpg')
fake = glob.glob('data/fake/*.jpg')

# If no fake images yet, simulate with augmented versions for demo
if len(fake) == 0:
    print('⚠️  No fake images found. Using augmented real as fake placeholder.')
    print('   Add your SimSwap-generated images to data/fake/ for real training.')
    fake = real[:len(real)//2]  # placeholder

all_paths = real + fake
all_labels = [0]*len(real) + [1]*len(fake)  # 0=REAL, 1=FAKE

tr_paths, va_paths, tr_labels, va_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

train_ds = DeepfakeDataset(tr_paths, tr_labels, train_tf)
val_ds = DeepfakeDataset(va_paths, va_labels, val_tf)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=16, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Step 4: Build EfficientNet-B4 model
import timm

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0, global_pool='avg')
        in_feat = self.backbone.num_features  # 1792
        self.attention = nn.Sequential(nn.Linear(in_feat, 512), nn.GELU(), nn.Dropout(0.3))
        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.attention(self.backbone(x))).squeeze(1)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

model = DeepfakeDetector().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Model loaded: {total_params/1e6:.1f}M parameters')

In [ ]:
# Step 5: Training loop
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

EPOCHS = 20
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()

best_val_acc = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs > 0.5)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.5
    scheduler.step()

    history['train_loss'].append(train_loss/len(train_dl))
    history['val_loss'].append(val_loss/len(val_dl))
    history['val_acc'].append(acc)
    history['val_auc'].append(auc)

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss/len(train_dl):.4f} | Val Acc: {acc:.4f} | AUC: {auc:.4f}')

    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(model.state_dict(), 'efficientnet_b4_ff++.pth')
        print(f'  💾 Saved best model (acc={acc:.4f})')

print(f'\n✅ Training complete. Best val acc: {best_val_acc:.4f}')

In [ ]:
# Step 6: Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train', color='#6366f1')
axes[0].plot(history['val_loss'], label='Val', color='#10b981')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_acc'], color='#6366f1')
axes[1].set_title('Val Accuracy')
axes[2].plot(history['val_auc'], color='#f59e0b')
axes[2].set_title('AUC-ROC')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 7: Download model checkpoint
from google.colab import files
files.download('efficientnet_b4_ff++.pth')
print('✅ Download the .pth file and place it at:')
print('   deepguard-ai/backend/models/efficientnet_b4_ff++.pth')

In [ ]:
# Step 8: (Optional) Push to HuggingFace Hub
HF_TOKEN = 'your_token_here'  # Replace
HF_REPO = 'your-username/deepguard-ai'  # Replace

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj='efficientnet_b4_ff++.pth',
    path_in_repo='efficientnet_b4_ff++.pth',
    repo_id=HF_REPO,
    repo_type='model',
)
print(f'✅ Model uploaded to https://huggingface.co/{HF_REPO}')